In [ ]:
import os
os.chdir('/mmgeneration')

In [ ]:
import os
import glob
import shutil
from pathlib import Path

import torch
import torch.nn.functional as F
from torchvision.utils import save_image
from PIL import Image, ImageOps
from tqdm import tqdm

from mmgen.apis import init_model, sample_img2img_model

# ========= CONFIGURA AQUÍ =========
config_file = 'configs/cyclegan/cyclegan_lsgan_resnet_ruta2arte_b1x1_270k_save_c_500_iter.py'

# Carpeta que contiene los checkpoints (p. ej. .../ckpt/.../iter_*.pth)
ckpt_glob = 'work_dirs/experiments/cyclegan_rutas2arte_save_c_500iter/ckpt/cyclegan_rutas2arte_save_c_500iter/iter_*.pth'

# Directorio de imágenes A (originales a estilizar)
input_dir = '/gan_aug/rutas2arte/trainA'

# Directorio de máscaras asociadas a las imágenes A (mismo basename)
masks_dir = 'automine1d/ann_dir/train'


# Carpeta raíz donde se guardará TODO (se crean subcarpetas por checkpoint)
output_root = 'outputs/stylized_by_ckpt_renamed'

# Nombre del dominio B EXACTO como en tu config
target_domain = 'trainB'

# Mantener la resolución original de la imagen de entrada en la salida (para alinear con máscara)
maintain_original_size = True

# (Opcional) Copiar también la imagen original a la salida (sin renombrar)
copy_originals = False

# Extensiones que consideraremos
img_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp')
mask_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
# =================================


def to_01(t: torch.Tensor) -> torch.Tensor:
    """Asegura rango [0,1] para guardar correctamente."""
    # t con forma (1, C, H, W)
    tmin = float(t.min())
    tmax = float(t.max())
    if tmin >= 0.0 and tmax <= 1.0:
        return t
    # Lo más típico: salida tanh en [-1,1]
    return t.add(1).div(2).clamp(0, 1)


def find_mask_for_image(masks_dir: str, img_path: Path):
    """Busca una máscara con el mismo basename que la imagen, con cualquiera de las extensiones válidas."""
    base = img_path.stem
    for ext in mask_exts:
        cand = Path(masks_dir) / f'{base}{ext}'
        if cand.exists():
            return cand
    return None


def main():
    os.makedirs(output_root, exist_ok=True)
    ckpt_list = sorted(glob.glob(ckpt_glob))
    if not ckpt_list:
        raise FileNotFoundError(f'No se encontraron checkpoints con patrón: {ckpt_glob}')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Recorre checkpoints
    for ckpt_path in ckpt_list:
        ckpt_path = Path(ckpt_path)
        ckpt_name = ckpt_path.stem  # p.ej. "iter_9000"

        # Subcarpetas por checkpoint
        out_dir_ckpt = Path(output_root) / ckpt_name
        out_imgs_dir = out_dir_ckpt / 'images'
        out_masks_dir = out_dir_ckpt / 'masks'
        out_dir_ckpt.mkdir(parents=True, exist_ok=True)
        out_imgs_dir.mkdir(exist_ok=True)
        out_masks_dir.mkdir(exist_ok=True)

        print(f'\n=== Procesando checkpoint: {ckpt_name} ===')

        # Inicializa modelo para este checkpoint
        model = init_model(str(config_file), str(ckpt_path), device=device)

        img_paths = [p for p in Path(input_dir).iterdir() if p.suffix.lower() in img_exts]
        if not img_paths:
            print(f'[WARN] No hay imágenes en {input_dir} con extensiones {img_exts}')
            continue

        for img_path in tqdm(img_paths, desc=f'{ckpt_name}', ncols=100):
            # Tamaño original (por si queremos mantenerlo)
            with Image.open(img_path) as im:
                im = ImageOps.exif_transpose(im)  # corrige orientación EXIF
                orig_w, orig_h = im.size

            # Genera estilizada A->B
            with torch.no_grad():
                result = sample_img2img_model(model, str(img_path), target_domain=target_domain)

            # Opcional: volver a tamaño original para alinear con máscara
            if maintain_original_size and (result.shape[-2] != orig_h or result.shape[-1] != orig_w):
                result = F.interpolate(result, size=(orig_h, orig_w), mode='bilinear', align_corners=False)

            # Nombre único para la estilizada: base__iter_XXXX__A2B.ext
            base = img_path.stem
            ext = img_path.suffix  # preserva extensión
            stylized_name = f'{base}__{ckpt_name}__A2B{ext}'
            out_img_path = out_imgs_dir / stylized_name

            # Guardar estilizada
            save_image(to_01(result).cpu(), str(out_img_path))

            # Copiar/renombrar máscara para que "matchee" con la estilizada
            mask_path = find_mask_for_image(masks_dir, img_path)
            if mask_path is not None:
                # Usa la extensión original de la máscara
                mask_ext = mask_path.suffix
                out_mask_path = out_masks_dir / (Path(stylized_name).stem + mask_ext)
                shutil.copy2(mask_path, out_mask_path)
            else:
                print(f'[WARN] No se encontró máscara para {img_path.name} en {masks_dir}')

            # (Opcional) copiar la imagen original a la carpeta del ckpt
            if copy_originals:
                shutil.copy2(img_path, out_imgs_dir / img_path.name)

        # Liberar memoria GPU entre checkpoints
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print('\n✅ Listo. Revisa:', output_root)


if __name__ == '__main__':
    main()


/home/maximilianovelez/miniconda3/envs/open-mmlab-mmgen/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
/home/maximilianovelez/miniconda3/envs/open-mmlab-mmgen/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/home/maximilianovelez/miniconda3/envs/open-mmlab-mmgen/lib/python3.8/site-packages/mmcv/cnn/bricks/conv_module.py:153: UserWarning: Unnecessary conv bias before batch/in


=== Procesando checkpoint: iter_1000 ===
load checkpoint from local path: work_dirs/experiments/cyclegan_rutas2arte_save_c_500iter/ckpt/cyclegan_rutas2arte_save_c_500iter/iter_1000.pth


iter_1000: 100%|████████████████████████████████████████████████████| 82/82 [01:49<00:00,  1.34s/it]



=== Procesando checkpoint: iter_10000 ===
load checkpoint from local path: work_dirs/experiments/cyclegan_rutas2arte_save_c_500iter/ckpt/cyclegan_rutas2arte_save_c_500iter/iter_10000.pth


iter_10000: 100%|███████████████████████████████████████████████████| 82/82 [01:42<00:00,  1.25s/it]


...


=== Procesando checkpoint: iter_9000 ===
load checkpoint from local path: work_dirs/experiments/cyclegan_rutas2arte_save_c_500iter/ckpt/cyclegan_rutas2arte_save_c_500iter/iter_9000.pth


iter_9000: 100%|████████████████████████████████████████████████████| 82/82 [01:42<00:00,  1.24s/it]



=== Procesando checkpoint: iter_9500 ===
load checkpoint from local path: work_dirs/experiments/cyclegan_rutas2arte_save_c_500iter/ckpt/cyclegan_rutas2arte_save_c_500iter/iter_9500.pth


iter_9500: 100%|████████████████████████████████████████████████████| 82/82 [01:42<00:00,  1.26s/it]


✅ Listo. Revisa: outputs/stylized_by_ckpt_renamed
